# 01 Bronze profiling

Download (or reuse) the verbatim source CSV, profile it, and confirm the
facts recorded during planning. Nothing is cleaned here.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

Insert the repo root on `sys.path` and pin SQLite.

In [ ]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ENV"] = "DEV"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.config.settings import get_settings

settings = get_settings()
print("env    :", settings.env)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

## Download and land the source file

The landing file is named by the sha256 of its bytes.

In [ ]:

from violence_etl.utils import io as etl_io

data = etl_io.download_source(settings.source_url)
path, sha, written = etl_io.write_landing(settings.bronze_dir, data)
print("bytes :", len(data))
print("sha256:", sha)
print("landing:", path)
print("written:", written)

## Profile the raw CSV

All fields stay text (`dtype=str`, `keep_default_na=False`).

In [ ]:
import pandas as pd

df = etl_io.read_csv(path)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.dtypes

In [ ]:
# Empty strings are preserved, not coerced to NaN.
empty_counts = (df == "").sum()
print("empty cells per column:")
print(empty_counts)

In [ ]:
# Expected planning facts: ZONA in {URBANA, RURAL}, CANTIDAD always 1,
# 32 departments, 777 municipality codes.
print(df["ZONA"].value_counts())
print("CANTIDAD unique:", df["CANTIDAD"].unique())
print("distinct departments:", df["COD_DEPTO"].nunique())
print("distinct municipalities:", df["COD_MUNI"].nunique())

In [ ]:
# Duplicate groups and date range (dates are DD/MM/YYYY, parsed explicitly).
fechas = pd.to_datetime(df["FECHA HECHO"], format="%d/%m/%Y", errors="coerce")
print("date min/max:", fechas.min(), fechas.max())
print("distinct full rows:", df.drop_duplicates().shape[0])
print("exact duplicate rows:", df.shape[0] - df.drop_duplicates().shape[0])
business = df.assign(fecha=fechas).drop_duplicates(
    ["fecha", "COD_DEPTO", "COD_MUNI", "ZONA", "CANTIDAD"]
)
print("distinct business rows:", business.shape[0])

## Ingest into bronze and inspect the mirrored table

In [ ]:
from violence_etl.transformations.bronze import ingest_bronze
from violence_etl.db.connection import make_engine

engine = make_engine(settings=settings)
result = ingest_bronze(engine, settings, csv_path=path)
result

In [ ]:
pd.read_sql("select * from bronze_crimes_raw order by row_num limit 5", engine)

In [ ]:
pd.read_sql(
    "select count(*) as raw_rows, count(distinct source_sha256) as shas "
    "from bronze_crimes_raw",
    engine,
)

The bronze table mirrors the source verbatim: 15,696 rows for the real
dataset, every source field stored as text, plus `source_url`, `source_file`,
`source_sha256`, `load_ts` and 1-based `row_num`.